In [0]:
%run ../log/00_audit_logging

In [0]:
# Databricks notebook source
# MAGIC %run ../log/audit_logging

# COMMAND ----------

# RAW -> BRONZE
# Reads raw TMDB JSON files (full or incremental) and stores every movie as a raw JSON string.
# Because the movie is stored as text, new/changed columns in the source never break this step.

from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, ArrayType
)

BRONZE = "workspace.tmdb.bronze_movies"
QUARANTINE = "workspace.tmdb.quarantine_movies"
RAW_BASE = "/Volumes/workspace/tmdb/raw"

# COMMAND ----------

# Parameters (change these in the widgets, no code edits needed)
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load type")
dbutils.widgets.text("source_path", "", "Folder or single .json file (blank = default folder)")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = auto timestamp)")

load_type   = dbutils.widgets.get("load_type")
source_path = dbutils.widgets.get("source_path").strip()
batch_id    = dbutils.widgets.get("batch_id").strip() or datetime.now().strftime("%Y%m%d_%H%M%S")

if source_path == "":
    source_path = f"{RAW_BASE}/{'full_load' if load_type == 'full' else 'incremental'}/"

# a folder -> read every .json inside it; a file -> read just that file
read_path = source_path if source_path.endswith(".json") else source_path.rstrip("/") + "/*.json"

print(f"load_type={load_type} | read_path={read_path} | batch_id={batch_id}")

# COMMAND ----------

# Explicit schemas (no inferSchema). "results" stays as strings = raw movie JSON.
full_schema = StructType([
    StructField("pulled_at",       StringType(),  True),
    StructField("source_endpoint", StringType(),  True),
    StructField("year_range",      StringType(),  True),
    StructField("enriched",        BooleanType(), True),
    StructField("enrich_limit",    LongType(),    True),
    StructField("record_count",    LongType(),    True),
    StructField("results",         ArrayType(StringType()), True),
    StructField("_corrupt",        StringType(),  True),   # bad files land here
])

incremental_schema = StructType([
    StructField("pulled_at", StringType(), True),
    StructField("trending_today", StructType([
        StructField("source_endpoint", StringType(), True),
        StructField("record_count",    LongType(),   True),
        StructField("results",         ArrayType(StringType()), True),
    ]), True),
    StructField("changed_movies", StructType([
        StructField("source_endpoint",        StringType(), True),
        StructField("changed_id_count",       LongType(),   True),
        StructField("resolved_record_count",  LongType(),   True),
        StructField("results",                ArrayType(StringType()), True),
    ]), True),
    StructField("_corrupt", StringType(), True),
])

# COMMAND ----------

# Create the Bronze table if needed
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE} (
    pulled_at         STRING,
    source_endpoint   STRING,
    source_collection STRING,
    source_file       STRING,
    raw_record        STRING,
    batch_id          STRING,
    load_timestamp    TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

start_time = datetime.now()

try:
    reader = (spark.read
              .option("multiLine", "true")
              .option("mode", "PERMISSIVE")
              .option("columnNameOfCorruptRecord", "_corrupt"))

    src_file = F.col("_metadata.file_path").alias("source_file")

    if load_type == "full":
        df = reader.schema(full_schema).json(read_path)
        ok = df.filter("_corrupt IS NULL")
        bronze_df = ok.select(
            "pulled_at", "source_endpoint",
            F.lit("full_load").alias("source_collection"),
            src_file,
            F.explode("results").alias("raw_record"),
        )
    else:
        df = reader.schema(incremental_schema).json(read_path)
        ok = df.filter("_corrupt IS NULL")
        trending = ok.select(
            "pulled_at",
            F.col("trending_today.source_endpoint").alias("source_endpoint"),
            F.lit("trending").alias("source_collection"),
            src_file,
            F.explode("trending_today.results").alias("raw_record"),
        )
        changed = ok.select(
            "pulled_at",
            F.col("changed_movies.source_endpoint").alias("source_endpoint"),
            F.lit("changed").alias("source_collection"),
            src_file,
            F.explode("changed_movies.results").alias("raw_record"),
        )
        bronze_df = trending.unionByName(changed)

    bronze_df = (bronze_df
                 .withColumn("batch_id", F.lit(batch_id))
                 .withColumn("load_timestamp", F.current_timestamp()))

    # Files that could not be parsed with the schema -> quarantine (batch keeps going)
    bad_df = df.filter("_corrupt IS NOT NULL").select(
        F.lit("Raw-to-Bronze").alias("layer"),
        F.lit(batch_id).alias("batch_id"),
        src_file,
        F.lit("File does not match expected schema / invalid JSON").alias("reject_reason"),
        F.coalesce(F.col("_corrupt"), F.col("pulled_at")).alias("raw_record"),
        F.current_timestamp().alias("load_timestamp"),
    )

    # Idempotent re-run: remove anything this batch_id wrote before, then write again
    spark.sql(f"DELETE FROM {BRONZE} WHERE batch_id = '{batch_id}'")
    spark.sql(f"DELETE FROM {QUARANTINE} WHERE layer = 'Raw-to-Bronze' AND batch_id = '{batch_id}'")

    bad_df.write.format("delta").mode("append").saveAsTable(QUARANTINE)
    bronze_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(BRONZE)

    rows_inserted = spark.table(BRONZE).filter(F.col("batch_id") == batch_id).count()
    rows_bad = spark.table(QUARANTINE).filter(
        (F.col("layer") == "Raw-to-Bronze") & (F.col("batch_id") == batch_id)).count()

    log_pipeline_execution("Raw-to-Bronze", load_type, batch_id, read_path,
                           start_time, datetime.now(), "SUCCESS",
                           rows_inserted=rows_inserted, rows_quarantined=rows_bad)

except Exception as e:
    log_pipeline_execution("Raw-to-Bronze", load_type, batch_id, read_path,
                           start_time, datetime.now(), "FAILURE", error_msg=e)
    raise

In [0]:
%sql
select count(*) from FROM workspace.tmdb.bronze_movies
--total rows 

In [0]:
%sql
SELECT source_file, 
       get_json_object(raw_record,'$.title') AS title,
       get_json_object(raw_record,'$.release_date') AS release_date
FROM workspace.tmdb.bronze_movies
WHERE get_json_object(raw_record,'$.id') = '427930'
--the duplicated one 